# 🥈 Silver Currency — Data Transformation

## Goal
Read currency data from the Bronze Delta Table, clean and validate the data, and load it into the Silver Delta Table.

## Flow
Bronze Delta Table → Cleaning → Data Quality → Silver Delta Table

## 1. Read Bronze Data

In [0]:
def read_bronze():
    df = spark.table("bronze.currency_rates")

    return df

## 2. Clean Data

In [0]:
from pyspark.sql.functions import col, to_date, upper, trim

def clean_data(df):
    df = df.withColumn(
        "date",
        to_date(col("date"))
    )

    df = df.withColumn(
        "base",
        upper(trim(col("base")))
    )

    df = df.withColumn(
        "quote",
        upper(trim(col("quote")))
    )

    df = df.withColumn(
        "rate",
        col("rate").cast("double")
    )

    df = df.dropDuplicates(
        ["date", "base", "quote"]
    )

    return df

## 3. Data Quality Check

In [0]:
from pyspark.sql.functions import col

def check_data_quality(df):
    invalid_rows = df.filter(
        col("date").isNull() |
        col("base").isNull() |
        col("quote").isNull() |
        col("rate").isNull() |
        (col("rate") <= 0)
    )

    if invalid_rows.count() > 0:
        return False

    return True

## 4. Write to Silver Delta Table

In [0]:
def write_silver(df):
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("silver.currency_rates")

## 5. Run Silver Pipeline

In [0]:
df = read_bronze()

df = clean_data(df)

quality = check_data_quality(df)

if quality == True:
    write_silver(df)
else:
    print("Data quality check failed")

In [0]:
%sql
SELECT *
FROM silver.currency_rates;